# Exercícios Aula 03: Leitura Inteligente de Documentos

Este notebook contém **5 exercícios práticos** para reforçar os principais conceitos apresentados nos notebooks da Aula 03 (`03_01`, `03_02` e `03_03`):

1. Pipeline básico de OCR com EasyOCR
2. Pré-processamento de scans ruins antes do OCR
3. Parse e Extract com IDP (LandingAI)
4. Extração direta de dados estruturados com VLM
5. Perguntas contextuais e cuidado com alucinações em VLM's

Complete os blocos marcados com `# TODO` em cada exercício. Use os notebooks `03_01`, `03_02` e `03_03` como referência sempre que precisar relembrar a sintaxe de alguma função.

**Observação:** os Exercícios 3 e 4 exigem chaves de API (LandingAI e OpenRouter, respectivamente). Consulte o [README.md](README.md) do repositório para saber como obtê-las.

In [ ]:
from IPython import get_ipython
if 'google.colab' in str(get_ipython()):
    print("Preparando ambiente Google Colab")
    !pip install opencv-python==5.0.0.93
    !pip install opencv-contrib-python==5.0.0.93
    !pip install easyocr
    !pip install landingai-ade
    !pip install openai
    !git clone https://github.com/pvoloshyn/curso-visao-computacional.git
    %cd curso-visao-computacional
else:
    pass

## Carregando bibliotecas

Além das bibliotecas que já usamos, vamos utilizar `easyocr` (OCR), `landingai_ade` (IDP) e `openai` (VLM via OpenRouter).

In [ ]:
from getpass import getpass
from pathlib import Path
import json
import base64
import mimetypes

import easyocr
from landingai_ade import LandingAIADE
from openai import OpenAI

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
# Indica ao notebook to render figures in-page.
%matplotlib inline
from IPython.display import Image, Markdown

## Preparação: OCR

Antes do Exercício 1, vamos instanciar o leitor do EasyOCR e reaproveitar as funções `apresentar_resultados()` e `extrair_texto()` do notebook `03_01`. Essa célula é apenas infraestrutura — não faz parte dos exercícios.

In [ ]:
# Instanciando o leitor OCR (reaproveitado do notebook 03_01)
reader = easyocr.Reader(
    ['pt', 'en'],
    gpu=False
)

def apresentar_resultados(img_rgb: np.ndarray, resultados: list, *, mostrar_texto: bool = False) -> None:
    img_draw = img_rgb.copy()

    for bbox, texto, confianca in resultados:
        pontos = np.array(bbox, dtype=np.int32)

        cv2.polylines(img_draw, [pontos], True, (0, 255, 0), 2)

        if mostrar_texto:
            x = pontos[0][0]
            y = pontos[0][1] - 10

            cv2.putText(
                img_draw,
                texto,
                (x, y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 0, 0),
                2
            )

    plt.figure(figsize=(15, 10))
    plt.imshow(img_draw)
    plt.axis('off')
    plt.show()

def extrair_texto(resultados: list) -> str:
    textos = [texto for _, texto, _ in resultados]
    return '\n'.join(textos)

print("Reader OCR pronto!")

## Exercício 1: Pipeline Básico de OCR com EasyOCR

**Conceito reforçado:** o EasyOCR só funciona corretamente com imagens no formato **RGB**, e o método `reader.readtext()` retorna uma lista de tuplas `(bbox, texto, confiança)` — não apenas o texto puro.

1. Leia a imagem `imagens/03/teste-ocr-2.jpg` com OpenCV e converta para RGB.
2. Rode `reader.readtext()` na imagem.
3. Apresente os resultados com bounding boxes e texto usando `apresentar_resultados(..., mostrar_texto=True)`.
4. Calcule e imprima a **confiança média** de todas as detecções (dica: `resultados` é uma lista de tuplas — a confiança é o terceiro elemento de cada uma).

In [ ]:
# 1. Leia a imagem 'imagens/03/teste-ocr-2.jpg' e converta para RGB


In [ ]:
# 2. Rode reader.readtext() na imagem


In [ ]:
# 3. Apresente os resultados com bounding boxes e texto


In [ ]:
# 4. Calcule e imprima a confiança média de todas as detecções


## Exercício 2: Melhorando Scans Ruins

**Conceito reforçado:** nem todo motor de OCR trata sozinho digitalizações de baixa qualidade — muitas vezes é necessário aplicar pré-processamento (binarização adaptativa, redução de ruído) antes do OCR.

Vamos reaproveitar a função `enhance()` do notebook `03_01`.

In [ ]:
def enhance(img, to_gray=True, normalize=True, sharpen_value=0, thres_radius=5, thres_constant=5, noise_reduction=0):
    """
    Aplica alguns filtros para melhorar scans ruins (reaproveitado do notebook 03_01)

    :param img: imagem a ser tratada
    :param to_gray: indica se a imagem deve ser convertida para escala de cinza
    :param normalize: indica se a imagem deve ser normalizada
    :param sharpen_value: valor de aumento de nitidez
    :param thres_radius: se maior que 0, aplica binarização adaptiva com o raio indicado
    :param thres_constant: valor de atenuação da binarização
    :param noise_reduction: valor de redução de ruido
    :returns: imagem tratada
    """
    mat = img.copy()

    if to_gray or thres_radius > 0:
        mat = cv2.cvtColor(mat, cv2.COLOR_RGB2GRAY)

    if normalize:
        cv2.normalize(mat, mat, 0, 255, cv2.NORM_MINMAX)

    if sharpen_value > 0:
        block_size = (sharpen_value * 2 + 1)**2
        blurred = cv2.GaussianBlur(mat, (block_size, block_size), 10.0)
        mat = cv2.addWeighted(mat, 1.5, blurred, -0.5, 0, mat)

    if thres_radius > 0:
        mat = cv2.adaptiveThreshold(mat, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY,
                                    thres_radius*2+1, thres_constant)

    if noise_reduction > 0:
        mat = cv2.medianBlur(mat, noise_reduction*2+1)

    if to_gray or thres_radius > 0:
        mat = cv2.cvtColor(mat, cv2.COLOR_GRAY2RGB)

    return mat

1. Rode o OCR diretamente na imagem `imagens/03/ocr_bad_scan.jpg`, **sem nenhum tratamento**, e conte quantas ocorrências (`len(resultados)`) foram reconhecidas.
2. Aplique a função `enhance()` na imagem variando os parâmetros `thres_radius`, `thres_constant` e `noise_reduction`.
3. Rode o OCR novamente na imagem tratada e conte as ocorrências.
4. Compare a quantidade de resultados e a confiança média antes e depois do pré-processamento. O tratamento ajudou?

In [ ]:
# 1. Rode o OCR sem tratamento na imagem 'imagens/03/ocr_bad_scan.jpg' e conte as ocorrências


In [ ]:
# 2. Aplique enhance() variando thres_radius, thres_constant e noise_reduction


In [ ]:
# 3. Rode o OCR na imagem tratada e conte as ocorrências


In [ ]:
# 4. Compare os resultados (quantidade de ocorrências e confiança média) e escreva sua conclusão:


## Preparação: IDP (LandingAI)

Antes do Exercício 3, vamos configurar o cliente da LandingAI e reaproveitar a função `apresentar_landingai_structure()` do notebook `03_02`. Você vai precisar de uma API Key da LandingAI (veja o [README.md](README.md)).

In [ ]:
API_KEY_LANDINGAI = getpass("Digite a API KEY da LandingAI")

client_idp = LandingAIADE(apikey=API_KEY_LANDINGAI)

def apresentar_landingai_structure(
    image_rgb: np.ndarray,
    structure,
    page_index: int = 0,
    draw_table_cells: bool = True,
    draw_labels: bool = True,
    thickness: int = 2
):
    COLORS = {
        'text':  (0, 255, 0),      # verde
        'table': (0, 120, 255),    # azul
        'logo':  (255, 0, 255),    # magenta
        'figure': (255, 255, 0),   # amarelo
        'default': (180, 180, 180)
    }

    img = image_rgb.copy()
    h, w = img.shape[:2]
    page = structure.children[page_index]

    for item in page.children:
        item_type = item.type

        if item_type == 'table_cell' and not draw_table_cells:
            continue

        box = item.grounding.box

        x1 = int(box.xmin * w)
        y1 = int(box.ymin * h)
        x2 = int(box.xmax * w)
        y2 = int(box.ymax * h)

        color = COLORS.get(item_type, COLORS['default'])

        cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)

        if draw_labels:
            label = item_type
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1)

            cv2.rectangle(img, (x1, y1 - th - 8), (x1 + tw + 8, y1), color, -1)
            cv2.putText(
                img, label, (x1 + 4, y1 - 4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA
            )

    plt.figure(figsize=(15, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

print("Cliente LandingAI pronto!")

## Exercício 3: Parse e Extract com LandingAI (IDP)

**Conceito reforçado:** o pipeline de IDP é dividido em duas etapas independentes: `parse()` (OCR estruturado, que retorna `markdown`, `structure` e `metadata`) e `extract()` (dados estruturados a partir de um **schema** definido por você).

1. Faça o `parse()` da imagem `imagens/03/modelo-nfe.jpg` usando `client_idp.v2.parse()`.
2. Apresente a estrutura detectada usando `apresentar_landingai_structure()`.
3. Defina um schema **simples e próprio** (por exemplo, apenas `numero`, `data_emissao` e `valor_total`) e rode o `extract()` usando o `markdown` obtido no parse.
4. Compare os valores extraídos com o que você vê na imagem original — houve algum erro?

In [ ]:
# 1. Faça o parse() da imagem 'imagens/03/modelo-nfe.jpg'


In [ ]:
# 2. Apresente a estrutura detectada


In [ ]:
# 3. Defina um schema simples e próprio, e rode o extract()


In [ ]:
# 4. Compare os valores extraídos com a imagem original e escreva sua conclusão:


## Preparação: VLM (OpenRouter)

Antes dos Exercícios 4 e 5, vamos configurar o cliente do OpenRouter e reaproveitar as funções `perguntar()` e `perguntar_imagem()` do notebook `03_03`. Você vai precisar de uma API Key do OpenRouter (veja o [README.md](README.md)).

In [ ]:
API_KEY_OPENROUTER = getpass("Digite a API KEY do OpenRouter")
OPEN_ROUTER_DEFAULT_MODEL = 'google/gemma-4-26b-a4b-it:free'

client_vlm = OpenAI(
    api_key=API_KEY_OPENROUTER,
    base_url="https://openrouter.ai/api/v1"
)

def image_to_data_url(image_path: str) -> str:
    """Converte uma imagem para Data URL compatível com OpenAI/OpenRouter."""
    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {image_path}")

    mime_type, _ = mimetypes.guess_type(image_path)
    if mime_type is None and str(image_path).endswith('.webp'):
        mime_type = "image/webp"

    if mime_type not in {"image/jpeg", "image/png", "image/webp"}:
        raise ValueError(f"Formato não suportado: {mime_type}")

    with open(image_path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")

    return f"data:{mime_type};base64,{encoded}"

def perguntar(prompt: str, model: str = None, temperature: float = 0.2):
    response = client_vlm.chat.completions.create(
        model=model or OPEN_ROUTER_DEFAULT_MODEL,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

def perguntar_imagem(image_path: str, prompt: str, model: str = None):
    response = client_vlm.chat.completions.create(
        model=model or OPEN_ROUTER_DEFAULT_MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": image_to_data_url(image_path)}}
                ]
            }
        ]
    )
    return response.choices[0].message.content

print("Cliente OpenRouter pronto!")

## Exercício 4: Extração Direta de Dados com VLM

**Conceito reforçado:** um VLM permite obter dados estruturados em uma única etapa (`Input > VLM > Resposta`), sem precisar de um pipeline de OCR + extração separado como no IDP.

1. Usando a mesma imagem do Exercício 3 (`imagens/03/modelo-nfe.jpg`), monte um prompt pedindo ao VLM para retornar, em JSON, os mesmos campos do schema que você definiu (`numero`, `data_emissao` e `valor_total`).
2. Rode `perguntar_imagem()` com essa imagem e esse prompt.
3. Compare o resultado do VLM com o resultado obtido pelo IDP no Exercício 3. Os valores concordam? Qual abordagem pareceu mais confiável para essa imagem?

In [ ]:
# 1. Monte um prompt pedindo os campos numero, data_emissao e valor_total em JSON


In [ ]:
# 2. Rode perguntar_imagem() com a imagem do Exercício 3 e esse prompt


In [ ]:
# 3. Compare com o resultado do IDP (Exercício 3) e escreva sua conclusão:


## Exercício 5: Perguntas Contextuais e Cuidado com Alucinações

**Conceito reforçado:** VLM's conseguem responder perguntas contextuais que vão além de uma simples extração de texto (algo que OCR e IDP não fazem), mas também podem **alucinar** — é fundamental validar criticamente as respostas antes de confiar nelas em produção.

1. Use a imagem `imagens/03/hidraulica.webp` ou `imagens/03/grafico.jpg` e formule **sua própria pergunta contextual** sobre ela (algo que exija interpretação, não apenas ler texto).
2. Rode `perguntar_imagem()` com sua pergunta.
3. Avalie criticamente a resposta: ela está correta? Em caso de dúvida, em que ponto desse fluxo você recomendaria adicionar uma validação humana (**HITL**)? Escreva sua conclusão em um comentário.

In [ ]:
# 1. Escolha uma imagem e formule sua própria pergunta contextual


In [ ]:
# 2. Rode perguntar_imagem() com sua pergunta


In [ ]:
# 3. Avalie criticamente a resposta e escreva sua conclusão sobre validação humana (HITL):
